In [4]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(99)

MEAN, STD = 1000, 200
N_SAMPLES = 10000
OUTLIER_RATIO = 0.05

amount = np.random.normal(MEAN, STD, N_SAMPLES)
n_out = int(N_SAMPLES * OUTLIER_RATIO)
out = np.random.uniform(5000, 10000, n_out)

all_amount = np.concatenate([amount, out])
np.random.shuffle(all_amount)

true_labels = np.array([0] * N_SAMPLES + [1] * n_out)
np.random.shuffle(true_labels)

def detect_out_zscore(data, threshold=3):
    z_scores = np.abs(stats.zscore(data))
    return z_scores > threshold

def detect_out_iqr(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (data < lower_bound) | (data > upper_bound)

predicted_3sigma = detect_out_zscore(all_amount)
predicted_iqr = detect_out_iqr(all_amount)

print("\n" + "="*80)
print("СРАВНЕНИЕ МЕТОДОВ ОБНАРУЖЕНИЯ АНОМАЛИЙ")
print("="*80)

print(f"\nПравило 3σ:")
print(f"  - Обнаружено: {predicted_3sigma.sum()}")
print(f"  - Процент: {predicted_3sigma.sum()/len(all_amount)*100:.2f}%")

print(f"\nМетод IQR:")
print(f"  - Обнаружено: {predicted_iqr.sum()}")
print(f"  - Процент: {predicted_iqr.sum()/len(all_amount)*100:.2f}%")

# Сравнение с реальными метками
print(f"\nТочность (3σ): {(predicted_3sigma == true_labels).mean()*100:.2f}%")
print(f"Точность (IQR): {(predicted_iqr == true_labels).mean()*100:.2f}%")


СРАВНЕНИЕ МЕТОДОВ ОБНАРУЖЕНИЯ АНОМАЛИЙ

Правило 3σ:
  - Обнаружено: 439
  - Процент: 4.18%

Метод IQR:
  - Обнаружено: 554
  - Процент: 5.28%

Точность (3σ): 91.40%
Точность (IQR): 90.34%
